# UNSW-NB15：CNN / RNN / Transformer × Source-only / DANN / MMD / CORAL / CORAL

這份 notebook 使用 UNSW-NB15 official training/testing split：

- Source domain：`UNSW_NB15_training-set.csv`，訓練時使用 feature + binary label。
- Target domain：`UNSW_NB15_testing-set.csv`，DANN/MMD 訓練時只使用 target feature，不使用 target label。
- Target label：只在最後 evaluation 與 attack category 分析使用。
- 不建立 validation split。

模型不是 MLP-style。所有架構都先把 tabular 欄位轉成 feature tokens：

- 每個 dense feature 是一個 token。
- `service` 是一個 categorical token。
- `state` 是一個 categorical token。
- CNN / RNN / Transformer 都直接從 token sequence 抽 representation。
- classifier/domain head 僅保留最後必要的 single-logit linear layer。

主要實驗拆成 6 個 code cells：

1. Source-only baseline：CNN / RNN / Transformer 訓練；CNN / RNN / Transformer 評估。
2. DANN：CNN / RNN / Transformer 訓練；CNN / RNN / Transformer 評估。
3. MMD：CNN / RNN / Transformer 訓練；CNN / RNN / Transformer 評估。

所有二分類 loss 與 DANN domain loss 都使用 `nn.BCEWithLogitsLoss`。


## 1. 匯入套件


In [ ]:
# ============================================================
# Cell 1. 匯入套件
# ============================================================

from __future__ import annotations

import math
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import torch
from torch import nn
from torch.autograd import Function
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

plt.style.use('seaborn-v0_8-whitegrid')


## 2. 實驗設定


In [ ]:
# ============================================================
# Cell 2. 實驗設定
# ============================================================

@dataclass
class Config:
    data_dir: Path = Path('data/UNSW_NB15')
    source_file: str = 'UNSW_NB15_training-set.csv'
    target_file: str = 'UNSW_NB15_testing-set.csv'
    csv_encoding: str = 'utf-8-sig'

    seed: int = 42
    max_rows_per_file: Optional[int] = None

    batch_size: int = 256
    num_epochs: int = 30
    learning_rate: float = 1e-3
    weight_decay: float = 1e-4
    max_train_batches_per_epoch: Optional[int] = 200

    architectures: Tuple[str, ...] = ('CNN', 'RNN', 'Transformer')
    methods: Tuple[str, ...] = ('Source-only', 'DANN', 'MMD', 'CORAL')

    # DANN 設定。
    domain_weight: float = 1.0
    use_dynamic_grl_lambda: bool = True

    # CORAL 設定。
    coral_weight: float = 1.0
    use_dynamic_coral_weight: bool = True

    # MMD 設定。
    mmd_weight: float = 1.0
    use_dynamic_mmd_weight: bool = True
    mmd_kernel_mul: float = 2.0
    mmd_kernel_num: int = 5
    mmd_fix_sigma: Optional[float] = None

    # WeightedRandomSampler：用 source label 做 class-balanced sampling。
    use_weighted_sampler: bool = True
    sampler_class_boosts: Tuple[float, float] = (1.0, 1.0)  # [normal, attack]
    sampler_num_samples_multiplier: float = 1.0

    # BCEWithLogitsLoss pos_weight = attack_loss_weight / normal_loss_weight。
    normal_loss_weight: float = 1.0
    attack_loss_weight: float = 2.0

    # 非 MLP-style tokenization：每個 tabular 欄位會先變成 token。
    token_dim: int = 64
    feature_dim: int = 128
    dropout: float = 0.20

    # CNN backbone：Conv1D over feature tokens。
    cnn_channels: Tuple[int, int] = (64, 128)

    # RNN backbone：Bi-GRU over feature tokens。
    rnn_hidden_dim: int = 64
    rnn_num_layers: int = 1

    # Transformer backbone：TransformerEncoder over feature tokens。
    transformer_nhead: int = 4
    transformer_num_layers: int = 2
    transformer_ff_dim: int = 128

    num_workers: int = 0
    anomaly_threshold: float = 0.50
    drop_constant_features: bool = True


config = Config()


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device('cuda')
    if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')


set_seed(config.seed)
device = get_device()
print(f'使用裝置：{device}')
config


## 3. 讀取 UNSW-NB15 並建立 Source / Target Domain


In [ ]:
# ============================================================
# Cell 3. 讀取資料並建立 source / target domain
# ============================================================

LABEL_COLUMN = 'label'
ATTACK_TYPE_COLUMN = 'attack_cat'
ID_COLUMN = 'id'
FILE_COLUMN = '__file'
SPLIT_COLUMN = '__split'
CATEGORICAL_COLUMNS = ['proto', 'service', 'state']


def resolve_data_dir(config: Config) -> Path:
    candidates = [
        Path(config.data_dir),
        Path.cwd() / config.data_dir,
        Path.cwd() / 'UNSW NB15 Archive',
        Path.cwd().parent / config.data_dir,
        Path.cwd() / 'UNSW NB15',
        Path.cwd() / 'UNSW NB15' / 'UNSW NB15 Archive',
        Path.cwd().parent / 'UNSW NB15' / 'UNSW NB15 Archive',
        Path.cwd(),
    ]
    needed = {config.source_file, config.target_file}
    for candidate in candidates:
        if not candidate.exists() or not candidate.is_dir():
            continue
        available = {path.name for path in candidate.glob('*.csv')}
        if needed.issubset(available):
            return candidate
    raise FileNotFoundError(f'找不到 UNSW-NB15 official train/test CSV，目前工作目錄：{Path.cwd()}')


def normalize_attack_cat(series: pd.Series) -> pd.Series:
    return (
        series.astype(str)
        .str.strip()
        .str.replace(r'\s+', ' ', regex=True)
        .replace({'Backdoors': 'Backdoor'})
    )


def load_unsw_file(path: Path, split_name: str, config: Config, random_state: int) -> pd.DataFrame:
    print(f'讀取 CSV：{path.name}')
    df = pd.read_csv(path, encoding=config.csv_encoding, low_memory=False)
    df.columns = df.columns.str.strip()
    missing = [col for col in [LABEL_COLUMN, ATTACK_TYPE_COLUMN] + CATEGORICAL_COLUMNS if col not in df.columns]
    if missing:
        raise ValueError(f'{path.name} 缺少必要欄位：{missing}')
    if config.max_rows_per_file is not None and len(df) > config.max_rows_per_file:
        df = df.sample(n=config.max_rows_per_file, random_state=random_state).reset_index(drop=True)
    df[ATTACK_TYPE_COLUMN] = normalize_attack_cat(df[ATTACK_TYPE_COLUMN])
    df[LABEL_COLUMN] = pd.to_numeric(df[LABEL_COLUMN], errors='raise').astype(np.int64)
    df[FILE_COLUMN] = path.name
    df[SPLIT_COLUMN] = split_name
    return df


DATA_DIR = resolve_data_dir(config)
print(f'資料目錄：{DATA_DIR}')

source_df = load_unsw_file(DATA_DIR / config.source_file, 'source_training', config, config.seed).reset_index(drop=True)
target_df = load_unsw_file(DATA_DIR / config.target_file, 'target_testing', config, config.seed + 100).reset_index(drop=True)

print(f'Source domain: {source_df.shape}  用於 supervised training')
print(f'Target domain: {target_df.shape}  label 不進 DANN/MMD loss，只做最後 evaluation')


## 4. Source / Target 分布檢查


In [ ]:
# ============================================================
# Cell 4. Source / Target 分布檢查
# ============================================================

print('Source label distribution:')
display(source_df[LABEL_COLUMN].value_counts().rename_axis('label').reset_index(name='count'))

print('Target label distribution，僅供最後 evaluation 使用：')
display(target_df[LABEL_COLUMN].value_counts().rename_axis('label').reset_index(name='count'))

print('Source attack_cat distribution:')
display(source_df[ATTACK_TYPE_COLUMN].value_counts().rename_axis('attack_cat').reset_index(name='count'))

print('Target attack_cat distribution:')
display(target_df[ATTACK_TYPE_COLUMN].value_counts().rename_axis('attack_cat').reset_index(name='count'))


## 5. UNSW-NB15 前處理器


In [ ]:
# ============================================================
# Cell 5. UNSW-NB15 前處理器
# ============================================================

PROTO_COLUMN = 'proto'
SERVICE_COLUMN = 'service'
STATE_COLUMN = 'state'


def make_one_hot_encoder() -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)


def normalize_category(series: pd.Series) -> pd.Series:
    return series.astype(str).str.strip().replace({'': '<UNK>'})


def build_vocab(series_list: Sequence[pd.Series]) -> Dict[str, int]:
    values = pd.concat(series_list, axis=0).pipe(normalize_category)
    categories = sorted(values.dropna().unique().tolist())
    return {'<UNK>': 0, **{category: idx + 1 for idx, category in enumerate(categories)}}


class UNSWPreprocessor:
    def __init__(
        self,
        label_cols: Sequence[str] = (ID_COLUMN, LABEL_COLUMN, ATTACK_TYPE_COLUMN, FILE_COLUMN, SPLIT_COLUMN),
        proto_col: str = PROTO_COLUMN,
        service_col: str = SERVICE_COLUMN,
        state_col: str = STATE_COLUMN,
        drop_constant_features: bool = True,
    ) -> None:
        self.label_cols = set(label_cols)
        self.proto_col = proto_col
        self.service_col = service_col
        self.state_col = state_col
        self.drop_constant_features = drop_constant_features
        self.numeric_cols: list[str] = []
        self.dropped_constant_cols: list[str] = []
        self.imputer: Optional[SimpleImputer] = None
        self.scaler: Optional[StandardScaler] = None
        self.proto_encoder: Optional[OneHotEncoder] = None
        self.service_vocab: Dict[str, int] = {}
        self.state_vocab: Dict[str, int] = {}
        self.proto_one_hot_dim: int = 0

    def fit(self, source_df: pd.DataFrame, target_df: pd.DataFrame) -> 'UNSWPreprocessor':
        categorical_cols = [self.proto_col, self.service_col, self.state_col]
        missing_source = [col for col in categorical_cols + [LABEL_COLUMN] if col not in source_df.columns]
        missing_target = [col for col in categorical_cols + [LABEL_COLUMN] if col not in target_df.columns]
        if missing_source:
            raise ValueError(f'Source 缺少必要欄位：{missing_source}')
        if missing_target:
            raise ValueError(f'Target 缺少必要欄位：{missing_target}')

        candidate_cols = [col for col in source_df.columns if col not in self.label_cols]
        candidate_numeric_cols = [col for col in candidate_cols if col not in categorical_cols]
        numeric_source = self._numeric_frame(source_df, candidate_numeric_cols)

        if self.drop_constant_features:
            nunique = numeric_source.nunique(dropna=True)
            self.dropped_constant_cols = sorted(nunique[nunique <= 1].index.tolist())
        else:
            self.dropped_constant_cols = []

        self.numeric_cols = [col for col in candidate_numeric_cols if col not in self.dropped_constant_cols]
        numeric_source = self._numeric_frame(source_df, self.numeric_cols)

        self.imputer = SimpleImputer(strategy='median')
        imputed = self.imputer.fit_transform(numeric_source)

        self.scaler = StandardScaler()
        self.scaler.fit(imputed)

        self.proto_encoder = make_one_hot_encoder()
        proto_fit_df = pd.DataFrame({
            self.proto_col: pd.concat([source_df[self.proto_col], target_df[self.proto_col]], ignore_index=True).pipe(normalize_category)
        })
        self.proto_encoder.fit(proto_fit_df)
        sample_encoded = self.proto_encoder.transform(proto_fit_df.iloc[:1])
        if hasattr(sample_encoded, 'toarray'):
            sample_encoded = sample_encoded.toarray()
        self.proto_one_hot_dim = int(sample_encoded.shape[1])

        self.service_vocab = build_vocab([source_df[self.service_col], target_df[self.service_col]])
        self.state_vocab = build_vocab([source_df[self.state_col], target_df[self.state_col]])
        return self

    def _numeric_frame(self, df: pd.DataFrame, columns: Sequence[str]) -> pd.DataFrame:
        missing_cols = [col for col in columns if col not in df.columns]
        if missing_cols:
            raise ValueError(f'資料缺少 numeric feature 欄位：{missing_cols[:10]}')
        numeric_df = df.loc[:, list(columns)].apply(pd.to_numeric, errors='coerce')
        numeric_df = numeric_df.replace([np.inf, -np.inf], np.nan)
        return numeric_df.astype(np.float32)

    def _encode_ids(self, df: pd.DataFrame, col: str, vocab: Dict[str, int]) -> np.ndarray:
        if col in df.columns:
            values = normalize_category(df[col])
        else:
            values = pd.Series(['<UNK>'] * len(df), index=df.index)
        return values.map(lambda value: vocab.get(value, 0)).to_numpy(dtype=np.int64)

    def transform(
        self,
        df: pd.DataFrame,
        with_labels: bool = True,
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray, Optional[np.ndarray]]:
        if self.imputer is None or self.scaler is None or self.proto_encoder is None:
            raise RuntimeError('Preprocessor 尚未 fit。')

        numeric_df = self._numeric_frame(df, self.numeric_cols)
        numeric_scaled = self.scaler.transform(self.imputer.transform(numeric_df)).astype(np.float32)

        proto_df = pd.DataFrame({
            self.proto_col: normalize_category(df[self.proto_col]) if self.proto_col in df.columns else '<UNK>'
        })
        proto_one_hot = self.proto_encoder.transform(proto_df)
        if hasattr(proto_one_hot, 'toarray'):
            proto_one_hot = proto_one_hot.toarray()
        proto_one_hot = proto_one_hot.astype(np.float32)

        dense_features = np.concatenate([numeric_scaled, proto_one_hot], axis=1).astype(np.float32)
        service_ids = self._encode_ids(df, self.service_col, self.service_vocab)
        state_ids = self._encode_ids(df, self.state_col, self.state_vocab)
        labels = df[LABEL_COLUMN].to_numpy(dtype=np.int64) if with_labels else None
        return dense_features, service_ids, state_ids, labels


preprocessor = UNSWPreprocessor(drop_constant_features=config.drop_constant_features)
preprocessor.fit(source_df, target_df)

source_dense, source_service_ids, source_state_ids, source_y = preprocessor.transform(source_df, with_labels=True)
target_dense, target_service_ids, target_state_ids, target_y = preprocessor.transform(target_df, with_labels=True)

print(f'numeric feature 數量：{len(preprocessor.numeric_cols)}')
print(f'proto one-hot dim：{preprocessor.proto_one_hot_dim}')
print(f'dense feature dim：{source_dense.shape[1]}')
print(f'service vocab size：{len(preprocessor.service_vocab)}')
print(f'state vocab size  ：{len(preprocessor.state_vocab)}')
print(f'移除 source 常數 numeric 欄位數量：{len(preprocessor.dropped_constant_cols)}')
print(preprocessor.dropped_constant_cols)
print(f'source_dense: {source_dense.shape}')
print(f'target_dense: {target_dense.shape}')


## 6. Dataset、Sampler、DataLoader


In [ ]:
# ============================================================
# Cell 6. Dataset、WeightedRandomSampler 與 DataLoader
# ============================================================

class UNSWDataset(Dataset):
    def __init__(
        self,
        dense_features: np.ndarray,
        service_ids: np.ndarray,
        state_ids: np.ndarray,
        labels: Optional[np.ndarray] = None,
    ) -> None:
        self.dense_features = torch.tensor(dense_features, dtype=torch.float32)
        self.service_ids = torch.tensor(service_ids, dtype=torch.long)
        self.state_ids = torch.tensor(state_ids, dtype=torch.long)
        self.labels = None if labels is None else torch.tensor(labels, dtype=torch.long)

    def __len__(self) -> int:
        return self.dense_features.shape[0]

    def __getitem__(self, index: int):
        if self.labels is None:
            return self.dense_features[index], self.service_ids[index], self.state_ids[index]
        return self.dense_features[index], self.service_ids[index], self.state_ids[index], self.labels[index]


source_train_dataset = UNSWDataset(source_dense, source_service_ids, source_state_ids, labels=source_y)
target_train_dataset = UNSWDataset(target_dense, target_service_ids, target_state_ids, labels=None)
source_eval_dataset = UNSWDataset(source_dense, source_service_ids, source_state_ids, labels=source_y)
target_eval_dataset = UNSWDataset(target_dense, target_service_ids, target_state_ids, labels=target_y)


def build_source_weighted_sampler(labels: np.ndarray, config: Config, seed: int) -> Tuple[WeightedRandomSampler, pd.DataFrame]:
    labels = np.asarray(labels, dtype=np.int64)
    class_counts = np.bincount(labels, minlength=2).astype(np.float64)
    boosts = np.asarray(config.sampler_class_boosts, dtype=np.float64)

    if boosts.shape[0] != 2:
        raise ValueError('sampler_class_boosts 必須是長度 2，例如 (1.0, 1.0)。')

    per_class_weight = boosts / np.maximum(class_counts, 1.0)
    sample_weights = per_class_weight[labels]
    num_samples = int(round(len(labels) * config.sampler_num_samples_multiplier))

    generator = torch.Generator()
    generator.manual_seed(seed)
    sampler = WeightedRandomSampler(
        weights=torch.as_tensor(sample_weights, dtype=torch.double),
        num_samples=num_samples,
        replacement=True,
        generator=generator,
    )

    diagnostic_df = pd.DataFrame({
        'class_id': [0, 1],
        'class_name': ['Normal', 'Attack'],
        'source_count': class_counts.astype(int),
        'boost': boosts,
        'per_sample_weight': per_class_weight,
        'expected_sampler_probability': boosts / boosts.sum(),
    })
    return sampler, diagnostic_df


def make_train_loaders(config: Config, run_seed: int) -> Tuple[DataLoader, DataLoader]:
    if config.use_weighted_sampler:
        source_sampler, _ = build_source_weighted_sampler(source_y, config, seed=run_seed)
        source_loader = DataLoader(
            source_train_dataset,
            batch_size=config.batch_size,
            sampler=source_sampler,
            shuffle=False,
            num_workers=config.num_workers,
            drop_last=False,
        )
    else:
        source_generator = torch.Generator().manual_seed(run_seed)
        source_loader = DataLoader(
            source_train_dataset,
            batch_size=config.batch_size,
            shuffle=True,
            generator=source_generator,
            num_workers=config.num_workers,
            drop_last=False,
        )

    target_generator = torch.Generator().manual_seed(run_seed + 10_000)
    target_loader = DataLoader(
        target_train_dataset,
        batch_size=config.batch_size,
        shuffle=True,
        generator=target_generator,
        num_workers=config.num_workers,
        drop_last=False,
    )
    return source_loader, target_loader


source_eval_loader = DataLoader(
    source_eval_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=config.num_workers,
    drop_last=False,
)

target_eval_loader = DataLoader(
    target_eval_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=config.num_workers,
    drop_last=False,
)

_, source_sampler_diagnostic_df = build_source_weighted_sampler(source_y, config, seed=config.seed)
print('Source WeightedRandomSampler 診斷：')
display(source_sampler_diagnostic_df)

example_source_loader, example_target_loader = make_train_loaders(config, run_seed=config.seed)
print(f'Source train batches: {len(example_source_loader)}')
print(f'Target train batches: {len(example_target_loader)}')
print(f'Source eval batches : {len(source_eval_loader)}')
print(f'Target eval batches : {len(target_eval_loader)}')


## 7. 非 MLP-style：Feature Token CNN / RNN / Transformer 模型架構


In [ ]:
# ============================================================
# Cell 7. Feature-token CNN / RNN / Transformer + DANN/MMD 模型架構
# ============================================================

class FeatureTokenizer(nn.Module):
    def __init__(
        self,
        dense_input_dim: int,
        num_service_embeddings: int,
        num_state_embeddings: int,
        token_dim: int,
        dropout: float,
    ) -> None:
        super().__init__()
        self.dense_input_dim = dense_input_dim
        self.token_dim = token_dim

        self.dense_weight = nn.Parameter(torch.empty(dense_input_dim, token_dim))
        self.dense_bias = nn.Parameter(torch.zeros(dense_input_dim, token_dim))
        nn.init.xavier_uniform_(self.dense_weight)

        self.service_embedding = nn.Embedding(num_service_embeddings, token_dim)
        self.state_embedding = nn.Embedding(num_state_embeddings, token_dim)
        self.feature_type_embedding = nn.Parameter(torch.zeros(1, dense_input_dim + 2, token_dim))
        self.dropout = nn.Dropout(dropout)

    @property
    def num_tokens(self) -> int:
        return self.dense_input_dim + 2

    def forward(
        self,
        dense_features: torch.Tensor,
        service_ids: torch.Tensor,
        state_ids: torch.Tensor,
    ) -> torch.Tensor:
        dense_tokens = dense_features.unsqueeze(-1) * self.dense_weight.unsqueeze(0) + self.dense_bias.unsqueeze(0)
        service_token = self.service_embedding(service_ids).unsqueeze(1)
        state_token = self.state_embedding(state_ids).unsqueeze(1)
        tokens = torch.cat([dense_tokens, service_token, state_token], dim=1)
        tokens = tokens + self.feature_type_embedding[:, :tokens.size(1), :]
        return self.dropout(tokens)


class CNNFeatureExtractor(nn.Module):
    def __init__(self, token_dim: int, feature_dim: int, dropout: float, channels: Tuple[int, int]) -> None:
        super().__init__()
        c1, c2 = channels
        self.conv = nn.Sequential(
            nn.Conv1d(token_dim, c1, kernel_size=5, padding=2),
            nn.BatchNorm1d(c1),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv1d(c1, c2, kernel_size=3, padding=1),
            nn.BatchNorm1d(c2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv1d(c2, c2, kernel_size=3, padding=1),
            nn.BatchNorm1d(c2),
            nn.GELU(),
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.projection = nn.Sequential(
            nn.Flatten(),
            nn.Linear(c2, feature_dim),
            nn.LayerNorm(feature_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, tokens: torch.Tensor) -> torch.Tensor:
        x = tokens.transpose(1, 2)
        x = self.pool(self.conv(x))
        return self.projection(x)


class RNNFeatureExtractor(nn.Module):
    def __init__(
        self,
        token_dim: int,
        feature_dim: int,
        dropout: float,
        hidden_dim: int,
        num_layers: int,
    ) -> None:
        super().__init__()
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim
        self.rnn = nn.GRU(
            input_size=token_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=True,
        )
        self.projection = nn.Sequential(
            nn.Linear(hidden_dim * 2, feature_dim),
            nn.LayerNorm(feature_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, tokens: torch.Tensor) -> torch.Tensor:
        _, hidden = self.rnn(tokens)
        hidden = hidden.reshape(self.num_layers, 2, tokens.size(0), self.hidden_dim)
        last_layer = hidden[-1]
        pooled = torch.cat([last_layer[0], last_layer[1]], dim=1)
        return self.projection(pooled)


class TransformerFeatureExtractor(nn.Module):
    def __init__(
        self,
        num_tokens: int,
        token_dim: int,
        feature_dim: int,
        dropout: float,
        nhead: int,
        num_layers: int,
        ff_dim: int,
    ) -> None:
        super().__init__()
        if token_dim % nhead != 0:
            raise ValueError('token_dim 必須可以被 transformer_nhead 整除。')
        self.cls_token = nn.Parameter(torch.zeros(1, 1, token_dim))
        self.position_embedding = nn.Parameter(torch.zeros(1, num_tokens + 1, token_dim))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=token_dim,
            nhead=nhead,
            dim_feedforward=ff_dim,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.projection = nn.Sequential(
            nn.LayerNorm(token_dim),
            nn.Linear(token_dim, feature_dim),
            nn.LayerNorm(feature_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, tokens: torch.Tensor) -> torch.Tensor:
        cls_tokens = self.cls_token.expand(tokens.size(0), -1, -1)
        tokens = torch.cat([cls_tokens, tokens], dim=1)
        tokens = tokens + self.position_embedding[:, :tokens.size(1), :]
        encoded = self.encoder(tokens)
        return self.projection(encoded[:, 0, :])


def make_backbone(architecture: str, num_tokens: int, config: Config) -> nn.Module:
    architecture_key = architecture.strip().lower()
    if architecture_key == 'cnn':
        return CNNFeatureExtractor(
            token_dim=config.token_dim,
            feature_dim=config.feature_dim,
            dropout=config.dropout,
            channels=config.cnn_channels,
        )
    if architecture_key == 'rnn':
        return RNNFeatureExtractor(
            token_dim=config.token_dim,
            feature_dim=config.feature_dim,
            dropout=config.dropout,
            hidden_dim=config.rnn_hidden_dim,
            num_layers=config.rnn_num_layers,
        )
    if architecture_key in {'transformer', 'tra'}:
        return TransformerFeatureExtractor(
            num_tokens=num_tokens,
            token_dim=config.token_dim,
            feature_dim=config.feature_dim,
            dropout=config.dropout,
            nhead=config.transformer_nhead,
            num_layers=config.transformer_num_layers,
            ff_dim=config.transformer_ff_dim,
        )
    raise ValueError(f'未知 architecture：{architecture}')


class FeatureClassifier(nn.Module):
    def __init__(
        self,
        architecture: str,
        dense_input_dim: int,
        num_service_embeddings: int,
        num_state_embeddings: int,
        config: Config,
    ) -> None:
        super().__init__()
        self.architecture = architecture
        self.tokenizer = FeatureTokenizer(
            dense_input_dim=dense_input_dim,
            num_service_embeddings=num_service_embeddings,
            num_state_embeddings=num_state_embeddings,
            token_dim=config.token_dim,
            dropout=config.dropout,
        )
        self.feature_extractor = make_backbone(architecture, self.tokenizer.num_tokens, config)
        self.class_classifier = nn.Linear(config.feature_dim, 1)

    def extract_features(
        self,
        dense_features: torch.Tensor,
        service_ids: torch.Tensor,
        state_ids: torch.Tensor,
    ) -> torch.Tensor:
        tokens = self.tokenizer(dense_features, service_ids, state_ids)
        return self.feature_extractor(tokens)

    def forward(
        self,
        dense_features: torch.Tensor,
        service_ids: torch.Tensor,
        state_ids: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        features = self.extract_features(dense_features, service_ids, state_ids)
        class_logits = self.class_classifier(features).squeeze(-1)
        return class_logits, features


class GradientReversalFunction(Function):
    @staticmethod
    def forward(ctx, x: torch.Tensor, lambda_grl: float) -> torch.Tensor:
        ctx.lambda_grl = lambda_grl
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output: torch.Tensor):
        return -ctx.lambda_grl * grad_output, None


class GradientReversalLayer(nn.Module):
    def forward(self, x: torch.Tensor, lambda_grl: float = 1.0) -> torch.Tensor:
        return GradientReversalFunction.apply(x, lambda_grl)


class DANNNetwork(FeatureClassifier):
    def __init__(self, architecture: str, dense_input_dim: int, num_service_embeddings: int, num_state_embeddings: int, config: Config) -> None:
        super().__init__(architecture, dense_input_dim, num_service_embeddings, num_state_embeddings, config)
        self.grl = GradientReversalLayer()
        self.domain_classifier = nn.Linear(config.feature_dim, 1)

    def forward(
        self,
        dense_features: torch.Tensor,
        service_ids: torch.Tensor,
        state_ids: torch.Tensor,
        lambda_grl: float = 1.0,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        features = self.extract_features(dense_features, service_ids, state_ids)
        class_logits = self.class_classifier(features).squeeze(-1)
        reversed_features = self.grl(features, lambda_grl=lambda_grl)
        domain_logits = self.domain_classifier(reversed_features).squeeze(-1)
        return class_logits, domain_logits, features


def make_model(architecture: str, method: str, config: Config, seed_offset: int = 0) -> nn.Module:
    set_seed(config.seed + seed_offset)
    model_cls = DANNNetwork if method == 'DANN' else FeatureClassifier
    return model_cls(
        architecture=architecture,
        dense_input_dim=source_dense.shape[1],
        num_service_embeddings=len(preprocessor.service_vocab),
        num_state_embeddings=len(preprocessor.state_vocab),
        config=config,
    ).to(device)


for architecture in config.architectures:
    preview_model = make_model(architecture, 'Source-only', config, seed_offset=999)
    print(f'{architecture} parameters: {sum(p.numel() for p in preview_model.parameters()):,}')


## 8. Loss 與共用訓練工具


In [ ]:
# ============================================================
# Cell 8. Loss 與共用訓練工具
# ============================================================

def infinite_loader(loader: DataLoader) -> Iterable:
    while True:
        for batch in loader:
            yield batch


def effective_num_batches(*loaders: DataLoader, config: Config) -> int:
    num_batches = max(len(loader) for loader in loaders)
    if config.max_train_batches_per_epoch is not None:
        num_batches = min(num_batches, config.max_train_batches_per_epoch)
    return max(num_batches, 1)


def paired_domain_batches(source_loader: DataLoader, target_loader: DataLoader, config: Config) -> Tuple[Iterable, int]:
    num_batches = effective_num_batches(source_loader, target_loader, config=config)
    source_iter = infinite_loader(source_loader)
    target_iter = infinite_loader(target_loader)
    return zip(source_iter, target_iter), num_batches


def move_labeled_batch(batch, device: torch.device):
    dense_features, service_ids, state_ids, labels = batch
    return (
        dense_features.to(device),
        service_ids.to(device),
        state_ids.to(device),
        labels.to(device),
    )


def move_unlabeled_batch(batch, device: torch.device):
    dense_features, service_ids, state_ids = batch
    return dense_features.to(device), service_ids.to(device), state_ids.to(device)


def make_source_class_criterion(config: Config, device: torch.device) -> nn.BCEWithLogitsLoss:
    pos_weight_value = config.attack_loss_weight / max(config.normal_loss_weight, 1e-12)
    pos_weight = torch.tensor(pos_weight_value, dtype=torch.float32, device=device)
    return nn.BCEWithLogitsLoss(pos_weight=pos_weight)


def source_batch_metrics(
    class_logits: torch.Tensor,
    labels: torch.Tensor,
    threshold: float = 0.50,
) -> Tuple[int, int, int, int]:
    pred = (torch.sigmoid(class_logits) >= threshold).long()
    correct = (pred == labels).sum().item()
    true_positive = ((pred == 1) & (labels == 1)).sum().item()
    positive_total = (labels == 1).sum().item()
    total = labels.size(0)
    return correct, true_positive, positive_total, total


def domain_batch_accuracy(src_domain_logits: torch.Tensor, tgt_domain_logits: torch.Tensor) -> Tuple[int, int]:
    src_pred = (torch.sigmoid(src_domain_logits) >= 0.50).long()
    tgt_pred = (torch.sigmoid(tgt_domain_logits) >= 0.50).long()
    src_correct = (src_pred == 0).sum().item()
    tgt_correct = (tgt_pred == 1).sum().item()
    total = src_pred.size(0) + tgt_pred.size(0)
    return src_correct + tgt_correct, total


def schedule_value(epoch: int, total_epochs: int, enabled: bool) -> float:
    if not enabled:
        return 1.0
    p = epoch / total_epochs
    return 2.0 / (1.0 + math.exp(-10.0 * p)) - 1.0


def gaussian_kernel_matrix(
    source_features: torch.Tensor,
    target_features: torch.Tensor,
    kernel_mul: float = 2.0,
    kernel_num: int = 5,
    fix_sigma: Optional[float] = None,
) -> torch.Tensor:
    total_features = torch.cat([source_features, target_features], dim=0)
    total0 = total_features.unsqueeze(0)
    total1 = total_features.unsqueeze(1)
    l2_distance = ((total0 - total1) ** 2).sum(2)

    if fix_sigma is not None:
        bandwidth = torch.as_tensor(fix_sigma, dtype=source_features.dtype, device=source_features.device)
    else:
        n_samples = total_features.shape[0]
        bandwidth = l2_distance.detach().sum() / max(n_samples * n_samples - n_samples, 1)

    bandwidth = bandwidth / (kernel_mul ** (kernel_num // 2))
    bandwidth_list = [bandwidth * (kernel_mul ** i) for i in range(kernel_num)]
    kernel_values = [torch.exp(-l2_distance / bw.clamp_min(1e-12)) for bw in bandwidth_list]
    return sum(kernel_values)


def mmd_loss(source_features: torch.Tensor, target_features: torch.Tensor, config: Config) -> torch.Tensor:
    """Multi-kernel MMD loss：對齊 source / target 特徵分布的一階高維 RKHS 平均表示。"""
    batch_size = min(source_features.size(0), target_features.size(0))
    source_features = source_features[:batch_size]
    target_features = target_features[:batch_size]

    kernels = gaussian_kernel_matrix(
        source_features,
        target_features,
        kernel_mul=config.mmd_kernel_mul,
        kernel_num=config.mmd_kernel_num,
        fix_sigma=config.mmd_fix_sigma,
    )

    xx = kernels[:batch_size, :batch_size]
    yy = kernels[batch_size:, batch_size:]
    xy = kernels[:batch_size, batch_size:]
    yx = kernels[batch_size:, :batch_size]
    return xx.mean() + yy.mean() - xy.mean() - yx.mean()

def coral_loss(feat_s: torch.Tensor, feat_t: torch.Tensor) -> torch.Tensor:
    """CORAL (CORrelation ALignment) loss：對齊 source / target 特徵的二階統計量（共變異數矩陣）。"""
    d = feat_s.size(1)

    def _covariance(feat: torch.Tensor) -> torch.Tensor:
        n = feat.size(0)
        feat_centered = feat - feat.mean(dim=0, keepdim=True)
        return feat_centered.t() @ feat_centered / (n - 1 + 1e-8)

    cov_s = _covariance(feat_s)
    cov_t = _covariance(feat_t)
    return (cov_s - cov_t).pow(2).sum() / (4 * d * d)


## 9. 三種方法訓練與評估函式


In [ ]:
# ============================================================
# Cell 9. Source-only / DANN / MMD / CORAL 訓練與評估函式
# ============================================================

def model_display_name(architecture: str, method: str) -> str:
    return f'{architecture} Source-only' if method == 'Source-only' else f'{architecture}-{method}'


def train_source_only(
    model: FeatureClassifier,
    architecture: str,
    source_loader: DataLoader,
    config: Config,
    device: torch.device,
) -> pd.DataFrame:
    class_criterion = make_source_class_criterion(config, device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
    history = []
    num_batches = effective_num_batches(source_loader, config=config)
    model_name = model_display_name(architecture, 'Source-only')

    for epoch in range(1, config.num_epochs + 1):
        model.train()
        running_loss = 0.0
        source_correct = 0
        source_true_positive = 0
        source_positive_total = 0
        source_total = 0

        for batch_idx, source_batch in enumerate(source_loader, start=1):
            src_dense, src_service, src_state, src_y = move_labeled_batch(source_batch, device)
            class_logits, _ = model(src_dense, src_service, src_state)
            class_loss = class_criterion(class_logits, src_y.float())

            optimizer.zero_grad(set_to_none=True)
            class_loss.backward()
            optimizer.step()

            running_loss += class_loss.item()
            correct, true_positive, positive_total, total = source_batch_metrics(class_logits, src_y)
            source_correct += correct
            source_true_positive += true_positive
            source_positive_total += positive_total
            source_total += total

            if batch_idx >= num_batches:
                break

        epoch_info = {
            'architecture': architecture,
            'method': 'Source-only',
            'model': model_name,
            'epoch': epoch,
            'total_loss': running_loss / num_batches,
            'class_loss': running_loss / num_batches,
            'domain_loss': np.nan,
            'mmd_loss': np.nan,
            'alignment_weight': 0.0,
            'domain_train_accuracy': np.nan,
            'source_train_accuracy': source_correct / max(source_total, 1),
            'source_train_recall': source_true_positive / max(source_positive_total, 1),
        }
        history.append(epoch_info)
        print(
            f"[{model_name}] Epoch {epoch:03d}/{config.num_epochs} | "
            f"loss={epoch_info['total_loss']:.4f} | "
            f"src_acc={epoch_info['source_train_accuracy']:.4f} | "
            f"src_recall={epoch_info['source_train_recall']:.4f}"
        )

    return pd.DataFrame(history)


def train_dann(
    model: DANNNetwork,
    architecture: str,
    source_loader: DataLoader,
    target_loader: DataLoader,
    config: Config,
    device: torch.device,
) -> pd.DataFrame:
    class_criterion = make_source_class_criterion(config, device)
    domain_criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
    history = []
    model_name = model_display_name(architecture, 'DANN')

    for epoch in range(1, config.num_epochs + 1):
        model.train()
        lambda_grl = schedule_value(epoch, config.num_epochs, config.use_dynamic_grl_lambda)
        paired_iter, num_batches = paired_domain_batches(source_loader, target_loader, config=config)

        running_total_loss = 0.0
        running_class_loss = 0.0
        running_domain_loss = 0.0
        domain_correct = 0
        domain_total = 0
        source_correct = 0
        source_true_positive = 0
        source_positive_total = 0
        source_total = 0

        for batch_idx, (source_batch, target_batch) in enumerate(paired_iter, start=1):
            src_dense, src_service, src_state, src_y = move_labeled_batch(source_batch, device)
            tgt_dense, tgt_service, tgt_state = move_unlabeled_batch(target_batch, device)

            src_domain_y = torch.zeros(src_y.size(0), dtype=torch.float32, device=device)
            tgt_domain_y = torch.ones(tgt_dense.size(0), dtype=torch.float32, device=device)

            src_class_logits, src_domain_logits, _ = model(src_dense, src_service, src_state, lambda_grl=lambda_grl)
            _, tgt_domain_logits, _ = model(tgt_dense, tgt_service, tgt_state, lambda_grl=lambda_grl)

            class_loss = class_criterion(src_class_logits, src_y.float())
            domain_loss = domain_criterion(src_domain_logits, src_domain_y) + domain_criterion(tgt_domain_logits, tgt_domain_y)
            total_loss = class_loss + config.domain_weight * domain_loss

            optimizer.zero_grad(set_to_none=True)
            total_loss.backward()
            optimizer.step()

            running_total_loss += total_loss.item()
            running_class_loss += class_loss.item()
            running_domain_loss += domain_loss.item()
            correct, true_positive, positive_total, total = source_batch_metrics(src_class_logits, src_y)
            source_correct += correct
            source_true_positive += true_positive
            source_positive_total += positive_total
            source_total += total
            d_correct, d_total = domain_batch_accuracy(src_domain_logits, tgt_domain_logits)
            domain_correct += d_correct
            domain_total += d_total

            if batch_idx >= num_batches:
                break

        epoch_info = {
            'architecture': architecture,
            'method': 'DANN',
            'model': model_name,
            'epoch': epoch,
            'total_loss': running_total_loss / num_batches,
            'class_loss': running_class_loss / num_batches,
            'domain_loss': running_domain_loss / num_batches,
            'mmd_loss': np.nan,
            'alignment_weight': lambda_grl,
            'domain_train_accuracy': domain_correct / max(domain_total, 1),
            'source_train_accuracy': source_correct / max(source_total, 1),
            'source_train_recall': source_true_positive / max(source_positive_total, 1),
        }
        history.append(epoch_info)
        print(
            f"[{model_name}] Epoch {epoch:03d}/{config.num_epochs} | "
            f"lambda={lambda_grl:.3f} | "
            f"total={epoch_info['total_loss']:.4f} | "
            f"class={epoch_info['class_loss']:.4f} | "
            f"domain={epoch_info['domain_loss']:.4f} | "
            f"domain_acc={epoch_info['domain_train_accuracy']:.4f} | "
            f"src_acc={epoch_info['source_train_accuracy']:.4f} | "
            f"src_recall={epoch_info['source_train_recall']:.4f}"
        )

    return pd.DataFrame(history)


def train_mmd(
    model: FeatureClassifier,
    architecture: str,
    source_loader: DataLoader,
    target_loader: DataLoader,
    config: Config,
    device: torch.device,
) -> pd.DataFrame:
    class_criterion = make_source_class_criterion(config, device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
    history = []
    model_name = model_display_name(architecture, 'MMD')

    for epoch in range(1, config.num_epochs + 1):
        model.train()
        mmd_weight = config.mmd_weight * schedule_value(epoch, config.num_epochs, config.use_dynamic_mmd_weight)
        paired_iter, num_batches = paired_domain_batches(source_loader, target_loader, config=config)

        running_total_loss = 0.0
        running_class_loss = 0.0
        running_mmd_loss = 0.0
        source_correct = 0
        source_true_positive = 0
        source_positive_total = 0
        source_total = 0

        for batch_idx, (source_batch, target_batch) in enumerate(paired_iter, start=1):
            src_dense, src_service, src_state, src_y = move_labeled_batch(source_batch, device)
            tgt_dense, tgt_service, tgt_state = move_unlabeled_batch(target_batch, device)

            src_class_logits, src_features = model(src_dense, src_service, src_state)
            _, tgt_features = model(tgt_dense, tgt_service, tgt_state)

            class_loss = class_criterion(src_class_logits, src_y.float())
            alignment_loss = mmd_loss(src_features, tgt_features, config)
            total_loss = class_loss + mmd_weight * alignment_loss

            optimizer.zero_grad(set_to_none=True)
            total_loss.backward()
            optimizer.step()

            running_total_loss += total_loss.item()
            running_class_loss += class_loss.item()
            running_mmd_loss += alignment_loss.item()
            correct, true_positive, positive_total, total = source_batch_metrics(src_class_logits, src_y)
            source_correct += correct
            source_true_positive += true_positive
            source_positive_total += positive_total
            source_total += total

            if batch_idx >= num_batches:
                break

        epoch_info = {
            'architecture': architecture,
            'method': 'MMD',
            'model': model_name,
            'epoch': epoch,
            'total_loss': running_total_loss / num_batches,
            'class_loss': running_class_loss / num_batches,
            'domain_loss': np.nan,
            'mmd_loss': running_mmd_loss / num_batches,
            'alignment_weight': mmd_weight,
            'domain_train_accuracy': np.nan,
            'source_train_accuracy': source_correct / max(source_total, 1),
            'source_train_recall': source_true_positive / max(source_positive_total, 1),
        }
        history.append(epoch_info)
        print(
            f"[{model_name}] Epoch {epoch:03d}/{config.num_epochs} | "
            f"mmd_w={mmd_weight:.3f} | "
            f"total={epoch_info['total_loss']:.4f} | "
            f"class={epoch_info['class_loss']:.4f} | "
            f"mmd={epoch_info['mmd_loss']:.4f} | "
            f"src_acc={epoch_info['source_train_accuracy']:.4f} | "
            f"src_recall={epoch_info['source_train_recall']:.4f}"
        )

    return pd.DataFrame(history)


def train_coral(
    model: FeatureClassifier,
    architecture: str,
    source_loader: DataLoader,
    target_loader: DataLoader,
    config: Config,
    device: torch.device,
) -> pd.DataFrame:
    class_criterion = make_source_class_criterion(config, device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
    history = []
    model_name = model_display_name(architecture, 'CORAL')

    for epoch in range(1, config.num_epochs + 1):
        model.train()
        coral_weight = config.coral_weight * schedule_value(epoch, config.num_epochs, config.use_dynamic_coral_weight)
        paired_iter, num_batches = paired_domain_batches(source_loader, target_loader, config=config)

        running_total_loss = 0.0
        running_class_loss = 0.0
        running_coral_loss = 0.0
        source_correct = 0
        source_true_positive = 0
        source_positive_total = 0
        source_total = 0

        for batch_idx, (source_batch, target_batch) in enumerate(paired_iter, start=1):
            src_dense, src_service, src_state, src_y = move_labeled_batch(source_batch, device)
            tgt_dense, tgt_service, tgt_state = move_unlabeled_batch(target_batch, device)

            src_class_logits, src_features = model(src_dense, src_service, src_state)
            _, tgt_features = model(tgt_dense, tgt_service, tgt_state)

            class_loss = class_criterion(src_class_logits, src_y.float())
            alignment_loss = coral_loss(src_features, tgt_features)
            total_loss = class_loss + coral_weight * alignment_loss

            optimizer.zero_grad(set_to_none=True)
            total_loss.backward()
            optimizer.step()

            running_total_loss += total_loss.item()
            running_class_loss += class_loss.item()
            running_coral_loss += alignment_loss.item()
            correct, true_positive, positive_total, total = source_batch_metrics(src_class_logits, src_y)
            source_correct += correct
            source_true_positive += true_positive
            source_positive_total += positive_total
            source_total += total

            if batch_idx >= num_batches:
                break

        epoch_info = {
            'architecture': architecture,
            'method': 'CORAL',
            'model': model_name,
            'epoch': epoch,
            'total_loss': running_total_loss / num_batches,
            'class_loss': running_class_loss / num_batches,
            'domain_loss': np.nan,
            'coral_loss': running_coral_loss / num_batches,
            'alignment_weight': coral_weight,
            'domain_train_accuracy': np.nan,
            'source_train_accuracy': source_correct / max(source_total, 1),
            'source_train_recall': source_true_positive / max(source_positive_total, 1),
        }
        history.append(epoch_info)
        print(
            f"[{model_name}] Epoch {epoch:03d}/{config.num_epochs} | "
            f"coral_w={coral_weight:.3f} | "
            f"total={epoch_info['total_loss']:.4f} | "
            f"class={epoch_info['class_loss']:.4f} | "
            f"coral={epoch_info['coral_loss']:.4f} | "
            f"src_acc={epoch_info['source_train_accuracy']:.4f} | "
            f"src_recall={epoch_info['source_train_recall']:.4f}"
        )

    return pd.DataFrame(history)


# ------------------------------------------------------------
# 評估與結果收集工具
# ------------------------------------------------------------

def get_class_logits_for_eval(model: nn.Module, dense_features, service_ids, state_ids) -> torch.Tensor:
    if isinstance(model, DANNNetwork):
        class_logits, _, _ = model(dense_features, service_ids, state_ids, lambda_grl=0.0)
        return class_logits
    class_logits, _ = model(dense_features, service_ids, state_ids)
    return class_logits


def collect_probs_and_labels(model: nn.Module, loader: DataLoader, device: torch.device) -> Tuple[np.ndarray, np.ndarray]:
    model.eval()
    y_true = []
    attack_probs = []
    with torch.no_grad():
        for batch in loader:
            dense_features, service_ids, state_ids, labels = move_labeled_batch(batch, device)
            class_logits = get_class_logits_for_eval(model, dense_features, service_ids, state_ids)
            probs = torch.sigmoid(class_logits)
            y_true.extend(labels.cpu().numpy().tolist())
            attack_probs.extend(probs.cpu().numpy().tolist())
    return np.asarray(y_true), np.asarray(attack_probs)


def parse_model_name(method_name: str) -> Tuple[str, str]:
    if method_name.endswith(' Source-only'):
        return method_name.replace(' Source-only', ''), 'Source-only'
    architecture, method = method_name.split('-', 1)
    return architecture, method


def evaluate_binary_metrics(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
    method_name: str,
    domain_name: str,
    threshold: float,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    y_true, attack_probs = collect_probs_and_labels(model, loader, device)
    y_pred = (attack_probs >= threshold).astype(np.int64)
    true_normal, false_positive, false_negative, true_positive = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    architecture, method = parse_model_name(method_name)

    metrics_df = pd.DataFrame([{
        'architecture': architecture,
        'method': method,
        'model': method_name,
        'domain': domain_name,
        'threshold': threshold,
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'true_normal': true_normal,
        'false_positive': false_positive,
        'false_negative': false_negative,
        'true_positive': true_positive,
    }])

    confusion_df = pd.DataFrame(
        [[true_normal, false_positive], [false_negative, true_positive]],
        index=['true_normal', 'true_attack'],
        columns=['pred_normal', 'pred_attack'],
    )
    return metrics_df, confusion_df


METRIC_COLUMNS = [
    'architecture', 'method', 'model', 'domain', 'threshold',
    'accuracy', 'precision', 'recall', 'f1',
    'true_normal', 'false_positive', 'false_negative', 'true_positive',
]

EXPERIMENT_RUN_IDS = {
    (architecture, method): arch_idx * len(config.methods) + method_idx + 1
    for arch_idx, architecture in enumerate(config.architectures)
    for method_idx, method in enumerate(config.methods)
}

trained_models: Dict[Tuple[str, str], nn.Module] = {}
history_tables_by_model: Dict[Tuple[str, str], pd.DataFrame] = {}
model_registry: Dict[str, nn.Module] = {}
model_thresholds: Dict[str, float] = {}
source_metrics_by_model: Dict[str, pd.DataFrame] = {}
target_metrics_by_model: Dict[str, pd.DataFrame] = {}
source_confusion_tables: Dict[str, pd.DataFrame] = {}
target_confusion_tables: Dict[str, pd.DataFrame] = {}

all_history_df = pd.DataFrame()
source_comparison_df = pd.DataFrame(columns=METRIC_COLUMNS)
target_comparison_df = pd.DataFrame(columns=METRIC_COLUMNS)
threshold_df = pd.DataFrame(columns=['model', 'chosen_threshold'])


def get_experiment_run_id(architecture: str, method_name: str) -> int:
    return EXPERIMENT_RUN_IDS[(architecture, method_name)]


def _concat_metric_tables(metric_tables: Dict[str, pd.DataFrame]) -> pd.DataFrame:
    if not metric_tables:
        return pd.DataFrame(columns=METRIC_COLUMNS)
    return (
        pd.concat(metric_tables.values(), ignore_index=True)
        .sort_values(['f1', 'model'], ascending=[False, True])
        .reset_index(drop=True)
    )


def _concat_history_for_method(method_name: str) -> pd.DataFrame:
    method_histories = [
        history_tables_by_model[(architecture, method_name)]
        for architecture in config.architectures
        if (architecture, method_name) in history_tables_by_model
    ]
    if not method_histories:
        return pd.DataFrame()
    return pd.concat(method_histories, ignore_index=True).reset_index(drop=True)


def refresh_threshold_df() -> pd.DataFrame:
    global threshold_df
    threshold_df = pd.DataFrame(
        [{'model': method_name, 'chosen_threshold': threshold} for method_name, threshold in model_thresholds.items()]
    )
    return threshold_df


def refresh_comparison_tables() -> Tuple[pd.DataFrame, pd.DataFrame]:
    global source_comparison_df, target_comparison_df
    source_comparison_df = _concat_metric_tables(source_metrics_by_model)
    target_comparison_df = _concat_metric_tables(target_metrics_by_model)
    return source_comparison_df, target_comparison_df


def register_training_result(
    architecture: str,
    method_name: str,
    model: nn.Module,
    history_df: pd.DataFrame,
) -> pd.DataFrame:
    global all_history_df
    key = (architecture, method_name)
    display_name = model_display_name(architecture, method_name)
    trained_models[key] = model
    history_tables_by_model[key] = history_df
    model_registry[display_name] = model
    model_thresholds[display_name] = config.anomaly_threshold
    all_history_df = pd.concat(history_tables_by_model.values(), ignore_index=True).reset_index(drop=True)
    refresh_threshold_df()
    return all_history_df


def evaluate_and_store_model(
    architecture: str,
    method_name: str,
    threshold: Optional[float] = None,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    key = (architecture, method_name)
    if key not in trained_models:
        raise KeyError(f'請先執行 {architecture} {method_name} 的訓練 cell。')

    display_name = model_display_name(architecture, method_name)
    threshold = model_thresholds.get(display_name, config.anomaly_threshold) if threshold is None else threshold
    trained_model = trained_models[key]

    source_metrics_df, source_confusion_df = evaluate_binary_metrics(
        model=trained_model,
        loader=source_eval_loader,
        device=device,
        method_name=display_name,
        domain_name='source',
        threshold=threshold,
    )
    target_metrics_df, target_confusion_df = evaluate_binary_metrics(
        model=trained_model,
        loader=target_eval_loader,
        device=device,
        method_name=display_name,
        domain_name='target_testing',
        threshold=threshold,
    )

    source_metrics_by_model[display_name] = source_metrics_df
    target_metrics_by_model[display_name] = target_metrics_df
    source_confusion_tables[display_name] = source_confusion_df
    target_confusion_tables[display_name] = target_confusion_df
    refresh_comparison_tables()
    return source_metrics_df, target_metrics_df, source_confusion_df, target_confusion_df


def evaluate_method_models(
    method_name: str,
    threshold: Optional[float] = None,
) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, pd.DataFrame], Dict[str, pd.DataFrame]]:
    source_metric_tables = []
    target_metric_tables = []
    source_confusions = {}
    target_confusions = {}

    for architecture in config.architectures:
        source_metrics_df, target_metrics_df, source_confusion_df, target_confusion_df = evaluate_and_store_model(
            architecture=architecture,
            method_name=method_name,
            threshold=threshold,
        )
        source_metric_tables.append(source_metrics_df)
        target_metric_tables.append(target_metrics_df)
        source_confusions[architecture] = source_confusion_df
        target_confusions[architecture] = target_confusion_df

    method_source_df = pd.concat(source_metric_tables, ignore_index=True).sort_values('f1', ascending=False).reset_index(drop=True)
    method_target_df = pd.concat(target_metric_tables, ignore_index=True).sort_values('f1', ascending=False).reset_index(drop=True)
    return method_source_df, method_target_df, source_confusions, target_confusions


## 10. Source-only baseline


In [ ]:
# ============================================================
# Cell 10a. Source-only baseline：CNN / RNN / Transformer 訓練
# ============================================================

method_name = 'Source-only'

for architecture in config.architectures:
    print('\n' + '=' * 72)
    print(f'{architecture} {method_name}')
    print('=' * 72)

    run_id = get_experiment_run_id(architecture, method_name)
    source_loader, _ = make_train_loaders(config, run_seed=config.seed + run_id * 100)
    model = make_model(architecture, method_name, config, seed_offset=run_id)
    history_df = train_source_only(model, architecture, source_loader, config, device)
    register_training_result(architecture, method_name, model, history_df)

source_only_history_df = _concat_history_for_method(method_name)
display(source_only_history_df.groupby('model').tail(1).reset_index(drop=True))


In [ ]:
# ============================================================
# Cell 10b. Source-only baseline：CNN / RNN / Transformer 評估
# ============================================================

method_name = 'Source-only'
(
    source_only_source_comparison_df,
    source_only_target_comparison_df,
    source_only_source_confusions,
    source_only_target_confusions,
) = evaluate_method_models(method_name)

print(f'Evaluation threshold = {config.anomaly_threshold:.2f}')
print('\nCurrent thresholds:')
display(threshold_df)
print('\nSource-only source metrics:')
display(source_only_source_comparison_df)
print('\nSource-only target metrics:')
display(source_only_target_comparison_df)

for architecture, confusion_df in source_only_target_confusions.items():
    print(f'\n{architecture} Source-only target confusion matrix:')
    display(confusion_df)

print('\nOverall target comparison so far:')
display(target_comparison_df)


## 11. DANN


In [ ]:
# ============================================================
# Cell 11a. DANN：CNN / RNN / Transformer 訓練
# ============================================================

method_name = 'DANN'

for architecture in config.architectures:
    print('\n' + '=' * 72)
    print(f'{architecture}-{method_name}')
    print('=' * 72)

    run_id = get_experiment_run_id(architecture, method_name)
    source_loader, target_loader = make_train_loaders(config, run_seed=config.seed + run_id * 100)
    model = make_model(architecture, method_name, config, seed_offset=run_id)
    history_df = train_dann(model, architecture, source_loader, target_loader, config, device)
    register_training_result(architecture, method_name, model, history_df)

dann_history_df = _concat_history_for_method(method_name)
display(dann_history_df.groupby('model').tail(1).reset_index(drop=True))


In [ ]:
# ============================================================
# Cell 11b. DANN：CNN / RNN / Transformer 評估
# ============================================================

method_name = 'DANN'
(
    dann_source_comparison_df,
    dann_target_comparison_df,
    dann_source_confusions,
    dann_target_confusions,
) = evaluate_method_models(method_name)

print(f'Evaluation threshold = {config.anomaly_threshold:.2f}')
print('\nCurrent thresholds:')
display(threshold_df)
print('\nDANN source metrics:')
display(dann_source_comparison_df)
print('\nDANN target metrics:')
display(dann_target_comparison_df)

for architecture, confusion_df in dann_target_confusions.items():
    print(f'\n{architecture}-DANN target confusion matrix:')
    display(confusion_df)

print('\nOverall target comparison so far:')
display(target_comparison_df)


## 12. MMD


In [ ]:
# ============================================================
# Cell 12a. MMD：CNN / RNN / Transformer 訓練
# ============================================================

method_name = 'MMD'

for architecture in config.architectures:
    print('\n' + '=' * 72)
    print(f'{architecture}-{method_name}')
    print('=' * 72)

    run_id = get_experiment_run_id(architecture, method_name)
    source_loader, target_loader = make_train_loaders(config, run_seed=config.seed + run_id * 100)
    model = make_model(architecture, method_name, config, seed_offset=run_id)
    history_df = train_mmd(model, architecture, source_loader, target_loader, config, device)
    register_training_result(architecture, method_name, model, history_df)

mmd_history_df = _concat_history_for_method(method_name)
display(mmd_history_df.groupby('model').tail(1).reset_index(drop=True))


In [ ]:
# ============================================================
# Cell 12b. MMD：CNN / RNN / Transformer 評估
# ============================================================

method_name = 'MMD'
(
    mmd_source_comparison_df,
    mmd_target_comparison_df,
    mmd_source_confusions,
    mmd_target_confusions,
) = evaluate_method_models(method_name)

print(f'Evaluation threshold = {config.anomaly_threshold:.2f}')
print('\nCurrent thresholds:')
display(threshold_df)
print('\nMMD source metrics:')
display(mmd_source_comparison_df)
print('\nMMD target metrics:')
display(mmd_target_comparison_df)

for architecture, confusion_df in mmd_target_confusions.items():
    print(f'\n{architecture}-MMD target confusion matrix:')
    display(confusion_df)

print('\nOverall target comparison so far:')
display(target_comparison_df)


## 13. CORAL


In [ ]:
# ============================================================
# Cell 13a. CORAL：CNN / RNN / Transformer 訓練
# ============================================================

method_name = 'CORAL'

for architecture in config.architectures:
    print('\n' + '=' * 72)
    print(f'{architecture}-{method_name}')
    print('=' * 72)

    run_id = get_experiment_run_id(architecture, method_name)
    source_loader, target_loader = make_train_loaders(config, run_seed=config.seed + run_id * 100)
    model = make_model(architecture, method_name, config, seed_offset=run_id)
    history_df = train_coral(model, architecture, source_loader, target_loader, config, device)
    register_training_result(architecture, method_name, model, history_df)

coral_history_df = _concat_history_for_method(method_name)
display(coral_history_df.groupby('model').tail(1).reset_index(drop=True))


In [ ]:
# ============================================================
# Cell 13b. CORAL：CNN / RNN / Transformer 評估
# ============================================================

method_name = 'CORAL'
(
    coral_source_comparison_df,
    coral_target_comparison_df,
    coral_source_confusions,
    coral_target_confusions,
) = evaluate_method_models(method_name)

print(f'Evaluation threshold = {config.anomaly_threshold:.2f}')
print('\nCurrent thresholds:')
display(threshold_df)
print('\nCORAL source metrics:')
display(coral_source_comparison_df)
print('\nCORAL target metrics:')
display(coral_target_comparison_df)

for architecture, confusion_df in coral_target_confusions.items():
    print(f'\n{architecture}-CORAL target confusion matrix:')
    display(confusion_df)

print('\nOverall target comparison so far:')
display(target_comparison_df)


## 13. Target attack_cat 分類別檢查


In [ ]:
# ============================================================
# Cell 13. Target attack_cat 分類別檢查
# ============================================================

if not model_registry:
    raise RuntimeError('請先完成至少一組訓練 cell。')


def predict_dataframe(
    model: nn.Module,
    dense_features: np.ndarray,
    service_ids: np.ndarray,
    state_ids: np.ndarray,
    original_df: pd.DataFrame,
    threshold: float,
    device: torch.device,
) -> pd.DataFrame:
    dataset = UNSWDataset(dense_features, service_ids, state_ids, labels=original_df[LABEL_COLUMN].to_numpy(dtype=np.int64))
    loader = DataLoader(dataset, batch_size=config.batch_size, shuffle=False, drop_last=False)
    y_true, probs = collect_probs_and_labels(model, loader, device)
    out = original_df[[ATTACK_TYPE_COLUMN, LABEL_COLUMN]].copy().reset_index(drop=True)
    out['attack_prob'] = probs
    out['prediction'] = (probs >= threshold).astype(np.int64)
    out['is_correct'] = out['prediction'].to_numpy() == y_true
    return out


def summarize_predictions(method_name: str, trained_model: nn.Module) -> pd.DataFrame:
    pred_df = predict_dataframe(
        trained_model,
        target_dense,
        target_service_ids,
        target_state_ids,
        target_df,
        threshold=model_thresholds[method_name],
        device=device,
    )
    architecture, method = parse_model_name(method_name)
    label_summary = (
        pred_df.groupby(ATTACK_TYPE_COLUMN)
        .agg(
            count=(LABEL_COLUMN, 'size'),
            true_attack=(LABEL_COLUMN, 'mean'),
            predicted_attack_rate=('prediction', 'mean'),
            accuracy=('is_correct', 'mean'),
            mean_attack_prob=('attack_prob', 'mean'),
        )
        .reset_index()
        .sort_values(ATTACK_TYPE_COLUMN)
    )
    label_summary.insert(0, 'method', method)
    label_summary.insert(0, 'architecture', architecture)
    label_summary.insert(0, 'model', method_name)
    return label_summary


per_label_comparison_df = pd.concat(
    [summarize_predictions(method_name, trained_model) for method_name, trained_model in model_registry.items()],
    ignore_index=True,
)
display(per_label_comparison_df)


## 14. 視覺化比較


In [ ]:
# ============================================================
# 視覺化比較
# ============================================================

if all_history_df.empty or target_comparison_df.empty:
    raise RuntimeError('請先至少完成一組訓練 cell 與對應的評估 cell。')

fig, axes = plt.subplots(2, 2, figsize=(15, 9), constrained_layout=True)
axes = axes.ravel()

for model_name, group in all_history_df.groupby('model'):
    axes[0].plot(group['epoch'], group['total_loss'], label=model_name)
    axes[1].plot(group['epoch'], group['source_train_recall'], label=model_name)

    if 'domain_train_accuracy' in group.columns and group['domain_train_accuracy'].notna().any():
        axes[2].plot(group['epoch'], group['domain_train_accuracy'], label=model_name)

    for loss_col, loss_label in [('mmd_loss', 'MMD'), ('coral_loss', 'CORAL')]:
        if loss_col in group.columns and group[loss_col].notna().any():
            axes[3].plot(group['epoch'], group[loss_col], label=f'{model_name} {loss_label}')

axes[0].set_title('Training total loss')
axes[0].set_xlabel('epoch')
axes[0].set_ylabel('loss')
axes[0].legend(fontsize=8, ncol=2)

axes[1].set_title('Source training attack recall')
axes[1].set_xlabel('epoch')
axes[1].set_ylabel('recall')
axes[1].set_ylim(0, 1.05)
axes[1].legend(fontsize=8, ncol=2)

axes[2].set_title('DANN domain classifier accuracy')
axes[2].set_xlabel('epoch')
axes[2].set_ylabel('accuracy')
axes[2].axhline(0.5, color='gray', linestyle='--', linewidth=1)
axes[2].set_ylim(0, 1.05)
if axes[2].lines:
    axes[2].legend(fontsize=8, ncol=2)
else:
    axes[2].text(0.5, 0.5, 'No domain accuracy logged', ha='center', va='center', transform=axes[2].transAxes)

axes[3].set_title('MMD / CORAL alignment loss')
axes[3].set_xlabel('epoch')
axes[3].set_ylabel('alignment loss')
if axes[3].lines:
    axes[3].legend(fontsize=8, ncol=2)
else:
    axes[3].text(0.5, 0.5, 'Run MMD or CORAL cells first', ha='center', va='center', transform=axes[3].transAxes)
plt.show()

metric_plot_df = target_comparison_df.set_index('model')[['accuracy', 'precision', 'recall', 'f1']]
ax = metric_plot_df.plot(kind='bar', figsize=(13, 5.2), ylim=(0, 1.05), rot=35)
ax.set_title('Target metrics comparison')
ax.set_ylabel('score')
ax.legend(loc='lower right')
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', padding=2, fontsize=7)
plt.tight_layout()
plt.show()

f1_pivot = target_comparison_df.pivot(index='architecture', columns='method', values='f1')
recall_pivot = target_comparison_df.pivot(index='architecture', columns='method', values='recall')

print('Target F1 pivot:')
display(f1_pivot)
print('Target recall pivot:')
display(recall_pivot)

all_history_df.tail()


## 如何解讀結果

- `Source-only` 是各架構 baseline，只使用 source labels，不使用 target features 做 adaptation。
- `DANN` 透過 Gradient Reversal 讓 feature extractor 學到 source/target 較難區分的表示；domain accuracy 越靠近 0.5，代表 domain classifier 越難分辨。
- `MMD` 用 multi-kernel MMD loss 直接縮小 source/target feature distribution 的距離；MMD loss 越低，代表 mini-batch feature distribution 越接近。
- `CORAL` 用 Deep CORAL loss 對齊 source/target feature covariance，計算比 MMD 輕量，適合觀察較小 domain shift 時的穩定性。
- 主要實驗已拆成 8 個 code cells：Source-only、DANN、MMD、CORAL 各自有訓練與評估 cell；每個訓練或評估 cell 都會一次處理 CNN / RNN / Transformer。
- `target_comparison_df` 是已完成評估的模型在 target domain 的主要比較表。
- 本版不是 MLP-style：模型不把整個 tabular vector 直接丟進多層 Linear feature extractor，而是先轉成 feature-token sequence，再由 Conv1D / GRU / TransformerEncoder 學 representation。
- 最後的 classifier/domain classifier 保留必要的 single-logit linear head，並全部使用 `nn.BCEWithLogitsLoss`。
